# Telemetry Sampling Analysis
Speed trace plots, sampled vs raw comparison.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from backend.database import engine

# Pick a specific driver + lap from the DB
query = """
SELECT ts.distance_m, ts.speed, ts.throttle, ts.brake, ts.gear
FROM telemetry_samples ts
JOIN drivers d ON d.id = ts.driver_id
JOIN sessions s ON s.id = ts.session_id
JOIN races r ON r.id = s.race_id
WHERE d.code = 'VER'
  AND r.name LIKE '%Bahrain%'
  AND s.session_type = 'R'
  AND ts.lap_number = 5
ORDER BY ts.distance_m
"""
df = pd.read_sql(query, engine)
print(f'Sampled points: {len(df)}')
df.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df['distance_m'], df['speed'], 'b-', linewidth=0.8)
axes[0].set_ylabel('Speed (km/h)')
axes[0].set_title('Sampled Telemetry — VER Bahrain Lap 5')

axes[1].plot(df['distance_m'], df['throttle'], 'g-', linewidth=0.8)
axes[1].set_ylabel('Throttle %')

axes[2].plot(df['distance_m'], df['brake'].astype(int), 'r-', linewidth=0.8)
axes[2].set_ylabel('Brake')
axes[2].set_xlabel('Distance (m)')

plt.tight_layout()
plt.show()

In [ ]:
# Compare sampled vs raw from FastF1
import fastf1
fastf1.Cache.enable_cache('data/fastf1_cache')

session = fastf1.get_session(2023, 1, 'R')
session.load()

ver_laps = session.laps.pick_drivers('VER')
lap5 = ver_laps[ver_laps['LapNumber'] == 5].iloc[0]
raw_tel = lap5.get_telemetry()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(raw_tel['Distance'], raw_tel['Speed'], 'b-', alpha=0.3, label=f'Raw ({len(raw_tel)} pts)')
ax.plot(df['distance_m'], df['speed'], 'r-', linewidth=1.5, label=f'Sampled 10m ({len(df)} pts)')
ax.set_xlabel('Distance (m)')
ax.set_ylabel('Speed (km/h)')
ax.legend()
ax.set_title('Raw vs 10m Sampled Telemetry')
plt.tight_layout()
plt.show()